# Deux lettres qui changent la lecture : CPIAUCSL ou CPIAUCNS · *Two letters that change the reading: CPIAUCSL or CPIAUCNS*

Notebook compagnon du chapitre **18. Atelier données : découvrir FRED, l'entrepôt de la Réserve fédérale** — [lire l'article](https://nmlab.io/ressources/atelier-donnees-decouvrir-fred).
Companion notebook to chapter **18. Data Workshop: Discovering FRED, the Federal Reserve's Warehouse** — [read the article](https://nmlab.io/en/ressources/data-workshop-discovering-fred).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


# données FRED chargées dans build_figure


from matplotlib.figure import Figure
import matplotlib.pyplot as plt
import pandas as pd

C, W = nm.COLORS, nm.WIDTH_PX


def wrap(ax, x: float, y: float, text: str, *, size: float = 19, color: str | None = None,
         weight: int = 500, ha: str = "left", va: str = "top", width: int = 42,
         lh: float = 1.5) -> int:
    """Écrit un texte replié à ``width`` caractères (coordonnées pixels)."""
    import textwrap
    lines: list[str] = []
    for para in text.split("\n"):
        lines += textwrap.wrap(para, width) or [""]
    ax.text(x, y, "\n".join(lines), fontsize=size, color=color or C["muted"],
            fontweight=weight, ha=ha, va=va, linespacing=lh, zorder=5)
    return len(lines)


def start(height: int = 1010) -> Figure:
    """Figure NMLab au format du site : 1747 px de large, fond sombre."""
    fig = nm.figure(height_px=height)
    fig.patch.set_facecolor(C["bg"])
    return fig


def dec(v: float, lang: str, n: int = 1, sign: bool = False) -> str:
    """Formate un nombre à la française (virgule, moins typographique) ou à l'anglaise."""
    s = f"{v:+.{n}f}" if sign else f"{v:.{n}f}"
    return s.replace("-", "−").replace(".", ",") if lang == "fr" else s


LABELS = {
    "fr": dict(
        title='Deux lettres qui changent la lecture : CPIAUCSL ou CPIAUCNS',
        sub='Le même indice des prix américain, corrigé ou non des variations saisonnières',
        l1='CPIAUCSL — désaisonnalisé',
        l2='CPIAUCNS — brut',
        msg="Variation d'un mois sur l'autre → série désaisonnalisée.\nGlissement sur douze mois → série brute.",
        note='BLS via FRED. Les deux séries racontent la même inflation ; prendre la mauvaise fausse\nla lecture mensuelle — le premier des six pièges du chapitre.',
    ),
    "en": dict(
        title='Two letters that change the reading: CPIAUCSL or CPIAUCNS',
        sub='The same US price index, seasonally adjusted or not',
        l1='CPIAUCSL — seasonally adjusted',
        l2='CPIAUCNS — not adjusted',
        msg='Month-on-month change → adjusted series.\nTwelve-month change → raw series.',
        note="BLS via FRED. Both series tell the same inflation story; picking the wrong one distorts\nthe monthly reading — the first of the chapter's six traps.",
    ),
}


def build_figure(lang: str = "fr") -> Figure:
    """Construit la figure NMLab (libellés selon ``lang``)."""
    t = LABELS[lang]
    sa = nm.load_fred("CPIAUCSL", "2022-01-01").pct_change() * 100
    ns = nm.load_fred("CPIAUCNS", "2022-01-01").pct_change() * 100
    sa, ns = sa.dropna(), ns.dropna()
    fig = start(1010); ax = nm.axes(fig, left=0.075, bottom=0.235)
    nm.header(fig, t["title"], t["sub"])
    ax.plot(ns.index, ns.values, color=C["rose"], lw=2.6, label=t["l2"], zorder=3)
    ax.plot(sa.index, sa.values, color=C["blue"], lw=3.2, label=t["l1"], zorder=4)
    ax.axhline(0, color=C["edge"], lw=1.6)
    leg = ax.legend(fontsize=18.5, frameon=True, facecolor=C["bg"], edgecolor=C["edge"],
                    loc="upper right", labelcolor=C["text"], ncol=2)
    leg.get_frame().set_linewidth(1.4)
    ax.set_ylabel("%", fontsize=18, color=C["muted"], labelpad=12)
    ax.grid(color=C["grid"], lw=1.1); ax.set_axisbelow(True)
    ax.tick_params(labelsize=17.5, colors=C["muted"], length=0)
    for sp in ("top", "right", "bottom", "left"): ax.spines[sp].set_visible(False)
    ax2 = nm.blank_axes(fig)
    ax2.text(W / 2, 196, t["msg"], fontsize=19.5, color=C["amber"], ha="center",
             va="center", fontweight=600, linespacing=1.6)
    nm.footer(fig, t["note"])
    return fig


build_figure(LANG)